1. Model: Qwen3-VL-8B vision with Unsloth (4-bit)

In [ ]:
!pip install -U "unsloth>=2024.10.0" "transformers>=4.57.0" datasets pillow accelerate bitsandbytes trl

2. Dataset: use your JSONL + real images

You already have JSONL rows like (simplified):

In [20]:
from datasets import load_dataset
from PIL import Image
import os

DATA_PATH   = "dataset.jsonl"   # your JSONL
IMAGES_ROOT = "."                           # root so that url is relative, change if needed

raw_ds = load_dataset(
    "json",
    data_files={"train": DATA_PATH},
)["train"]


Generating train split: 0 examples [00:00, ? examples/s]

Convert messages into Unsloth vision format

In [22]:
from PIL import Image
import os


from datasets import load_dataset
from PIL import Image
import os

IMAGE_ROOT = ""


def convert_for_vision(example):
    new_messages = []

    for msg in example["messages"]:
        role = msg["role"]
        blocks = []

        for block in msg["content"]:
            b_type = block.get("type")

            if b_type == "text":
                blocks.append({
                    "type": "text",
                    "text": block["text"],
                })

            elif b_type == "image_url":
                rel_path = block["image_url"]["url"]
                img_path = os.path.join(IMAGE_ROOT, rel_path)

                try:
                    image = Image.open(img_path).convert("RGB")
                except Exception as e:
                    print(f"⚠️ Could not open image: {img_path} error:", e)
                    image = Image.new("RGB", (512, 512), "white")

                blocks.append({
                    "type": "image",
                    "image": image,
                })

            else:
                blocks.append({
                    "type": "text",
                    "text": str(block),
                })

        new_messages.append({
            "role": role,
            "content": blocks,
        })

    return {"messages": new_messages}

converted_dataset = [convert_for_vision(ex) for ex in raw_ds]
print(len(converted_dataset), "examples")

221 examples


In [21]:
print(converted_dataset[0]["messages"][2])
print(raw_ds[1]['messages'][1])
ex = converted_dataset[0]
for msg in ex["messages"]:
    print("ROLE:", msg["role"])
    for block in msg["content"]:
        print("   ", block["type"], type(block.get("image") or block.get("text")))


{'role': 'assistant', 'content': [{'type': 'text', 'text': '{"PatientSurname": {"value": "T108", "bbox": {"x": 72.7, "y": 510.2, "width": 313.1, "height": 52.2}}, "PatientForeName": {"value": "T108", "bbox": {"x": 396.2, "y": 515.5, "width": 326.1, "height": 49.6}}, "DateOfBirth": {"value": "1/5/61", "bbox": {"x": 67.5, "y": 567.7, "width": 407, "height": 52.2}}, "Gender": {"value": "fEMale", "bbox": {"x": 490.1, "y": 562.4, "width": 349.6, "height": 54.9}}, "Ethnicity": {"value": "Caucasian", "bbox": {"x": 67.5, "y": 612, "width": 393.9, "height": 54.8}}, "Address": {"value": "108 Green Street", "bbox": {"x": 62.3, "y": 666.8, "width": 448.7, "height": 54.9}}, "HospitalNumber": {"value": "0000108", "bbox": {"x": 64.9, "y": 729.5, "width": 467, "height": 36.5}}, "NHSNumber": {"value": "", "bbox": {"x": 62.3, "y": 773.9, "width": 513.9, "height": 46.9}}, "LandlineNumber": {"value": "", "bbox": {"x": 67.5, "y": 826.1, "width": 493.1, "height": 46.9}}, "MobileNumber": {"value": "079364111

3. Supervised fine-tuning (SFT) with vision collator

Unsloth provides a vision data collator + standard SFTTrainer for this.

In [23]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))


CUDA available: False


In [ ]:
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# 1) Load model (Qwen3-VL 8B 4-bit)
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

# 2) Add LoRA
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    random_state= 3407,
    use_rslora  = False,
    loftq_config= None,
    target_modules = "all-linear",
)

param_device = next(model.parameters()).device
print("Model is on:", param_device)

# 3) Data collator
data_collator = UnslothVisionDataCollator(model, tokenizer)

# 4) Training config
train_args = SFTConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps= 4,
    learning_rate              = 2e-4,
    max_steps                  = 800,      # bump once it's stable
    warmup_ratio               = 0.03,
    logging_steps              = 10,
    save_steps                 = 200,
    weight_decay               = 0.01,
    bf16                       = is_bf16_supported(),
    output_dir                 = "qwen3-vl-8b-referral-forms",
    remove_unused_columns      = False,
    max_seq_length             = 2048,
    dataset_text_field         = "",       # important for vision with custom collator
    dataset_kwargs             = {"skip_prepare_dataset": True},
)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = data_collator,
    train_dataset = converted_dataset,   # <-- plain list, NOT HF Dataset
    args          = train_args,
)

trainer.train()


In [ ]:
# watch -n 1 nvidia-smi
# import torch
# print("CUDA:", torch.cuda.is_available())
# !nvidia-smi
# print("Model device:", next(model.parameters()).device)


This will fine-tune only the LoRA adapters while keeping Qwen3-VL’s vision + language backbone frozen, in 4-bit, which is what lets this run on modest GPUs.

4. Inference on a new referral form

Once training is done:

In [ ]:
from PIL import Image
import torch

FastVisionModel.for_inference(model)

def run_inference(image_path, instruction):
    image = Image.open(image_path).convert("RGB")

    # Build a single-turn vision chat
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": instruction},
            ],
        }
    ]

    # Qwen3-VL template is already inside tokenizer from Unsloth 
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,
    )

    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 512,
            temperature     = 0.2,
            top_p           = 0.8,
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

test_json = run_inference(
    "dataset/images_by_page/page_1_1/gp-referral-cancer-colorectal IOV108_page_1.png",
    "Extract the referral form fields and return STRICT valid JSON only."
)
print(test_json)


You can then json.loads(test_json) and validate / post-process.

5. Saving the fine-tuned Qwen3-VL model
Save LoRA adapters

In [ ]:
adapter_dir = "qwen3-vl-8b-referral-forms-lora"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved LoRA adapters to", adapter_dir)

(Optional) Merge LoRA into full weights for deployment

In [ ]:
merged_dir = "qwen3-vl-8b-referral-forms-merged"

# merged = FastVisionModel.merge_and_unload(model)
model.save_pretrained_merged(merged_dir, tokenizer)
# tokenizer.save_pretrained(merged_dir)

print("Saved merged model to", merged_dir)
model.push_to_hub_merged("sophy/finetuned-qwen-referrals", tokenizer, token="")

# Save to 16bit GGUF
model.save_pretrained_gguf("unsloth_finetune_referrals", tokenizer, quantization_method = "f16")
model.push_to_hub_gguf("hf/unsloth_finetune_referrals", tokenizer, quantization_method = "f16", token = "")


The merged model can be served with vLLM, llama.cpp (via GGUF exports), Docker, etc. – the same infra Unsloth shows for Qwen3-VL.

### Merge Lora

In [ ]:
from unsloth import FastVisionModel

BASE_MODEL   = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
ADAPTER_REPO = "sophy/finetuned-qwen-referrals"

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit = True,
)

model.load_adapter(ADAPTER_REPO)
FastVisionModel.for_inference(model)
print("✅ Base + LoRA ready to merge")


In [ ]:
merged_dir = "qwen3-vl-8b-referral-forms"

model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method = "merged",   # <-- full fp16/bf16 weights + config.json
)

print("✅ Saved merged model to", merged_dir)



In [ ]:
model.push_to_hub_merged("sophy/qwen3-vl-8b-referral-forms", tokenizer, token="")

# Save to 16bit GGUF
model.save_pretrained_gguf("unsloth_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16")
model.push_to_hub_gguf("hf/unsloth_finetune_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16", token = "")

## Test

In [ ]:
from unsloth import FastVisionModel

model_id = "sophy/qwen3-vl-8b-referral-forms"

model, tokenizer = FastVisionModel.from_pretrained(
    model_id,
    load_in_4bit = False,   # merged is usually fp16/bf16
)
FastVisionModel.for_inference(model)

In [ ]:
from PIL import Image
import torch

image_path = "images/a.png"
image = Image.open(image_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Extract all fields and return JSON."},
        ],
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)

inputs = tokenizer(image, prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))
